<a href="https://colab.research.google.com/github/Priyanka777444/unisign-sign-language-recognition/blob/main/Model_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall mediapipe -y
!pip install mediapipe --upgrade

In [ ]:
!pip install tensorflow

In [ ]:
!pip install "ml_dtypes>=0.5.0"

In [ ]:
import os
import cv2
import json
import numpy as np
import mediapipe as mp
from tensorflow.keras.layers import Conv3D
from tensorflow.keras.layers import TimeDistributed
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, MaxPooling3D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
base_url = '/content/drive/MyDrive/Classroom/datasets'
valid_extensions = ('.mp4', '.avi', '.mov')

# Load dataset
class_names, video_links, class_labels = [], [], []
for class_name in os.listdir(base_url):
    class_dir = os.path.join(base_url, class_name)
    if os.path.isdir(class_dir):
        class_names.append(class_name)
        class_index = len(class_names) - 1
        for filename in os.listdir(class_dir):
            if filename.lower().endswith(valid_extensions):
                video_links.append(os.path.join(class_dir, filename))
                class_labels.append(class_index)

print(f"Found {len(video_links)} videos across {len(class_names)} classes.")
print("Classes:", class_names)
print("Video count per class:", [class_labels.count(i) for i in range(len(class_names))])

# Save class label mapping
with open("class_labels.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False)

num_classes = len(class_names)
if num_classes == 0:
    raise ValueError("No classes found in the dataset directory.")

# Utility functions
def get_video_duration_and_fps(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Error opening video file: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    cap.release()
    return duration, fps, total_frames

def video_frame_generator(video_path, target_fps=30, max_frames=10):
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
    duration, original_fps, total_frames = get_video_duration_and_fps(video_path)
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)
    processed_frames = []

    cap = cv2.VideoCapture(video_path)
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            x_coords = [lm.x for lm in hand_landmarks.landmark]
            y_coords = [lm.y for lm in hand_landmarks.landmark]
            x_min = int(min(x_coords) * frame.shape[1]) - 20
            x_max = int(max(x_coords) * frame.shape[1]) + 20
            y_min = int(min(y_coords) * frame.shape[0]) - 20
            y_max = int(max(y_coords) * frame.shape[0]) + 20
            x_min, y_min = max(0, x_min), max(0, y_min)
            x_max, y_max = min(frame.shape[1], frame.shape[0]), min(frame.shape[0], frame.shape[1])
            hand_roi = frame[y_min:y_max, x_min:x_max]
            if hand_roi.size > 0:
                canny = cv2.Canny(hand_roi, 100, 200)
                img_resized = cv2.resize(canny, (128, 128))
                processed_frames.append(img_resized)
    cap.release()
    hands.close()

    # Padding if needed
    while len(processed_frames) < max_frames:
        pad = processed_frames[-1] if processed_frames else np.zeros((128, 128), dtype=np.uint8)
        processed_frames.append(pad)

    return np.array(processed_frames[:max_frames])

def preprocess_video_for_model(video_path, max_frames=10):
    try:
        frames = video_frame_generator(video_path, max_frames=max_frames)
        if frames.shape[0] != max_frames:
            return None
        frames = frames[..., np.newaxis] / 255.0  # Normalize and add channel
        return frames
    except Exception as e:
        print(f"[Warning] Skipping {video_path}: {e}")
        return None

# Prepare dataset
X_data, y_data = [], []
for path, label in zip(video_links, class_labels):
    X = preprocess_video_for_model(path, max_frames=10)
    if X is not None:
        X_data.append(X)
        y_data.append(label)

if not X_data:
    raise ValueError("No valid videos were processed.")

X_data = np.array(X_data)
y_data = to_categorical(np.array(y_data), num_classes=num_classes)
print("Final shape:", X_data.shape)

# Split into train/val
X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# Define ConvLSTM model_rnn
model_rnn = Sequential()
model_rnn.add(Conv3D(32, (3, 3, 3), activation='relu', padding='same', input_shape=(10, 128, 128, 1)))
model_rnn.add(MaxPooling3D(pool_size=(2, 2, 2)))
model_rnn.add(TimeDistributed(Flatten()))
model_rnn.add(LSTM(128))
model_rnn.add(Dense(num_classes, activation='softmax'))

model_rnn.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train
history = model_rnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=16)

# Save model_rnn
model_rnn.save("video.h5")

# Evaluation
train_loss, train_acc = model_rnn.evaluate(X_train, y_train)
val_loss, val_acc = model_rnn.evaluate(X_val, y_val)
print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

In [ ]:
from tensorflow.keras.models import load_model

def predict_video(video_path, model, max_frames=10):
    X = preprocess_video_for_model(video_path, max_frames)
    if X is None:
        print("Invalid input video.")
        return
    X = np.expand_dims(X, axis=0)
    prediction = model.predict(X)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction)
    with open("class_labels.json", "r", encoding="utf-8") as f:
        labels = json.load(f)
    print(f"Predicted class: {labels[class_index]} ({confidence * 100:.2f}%)")

model = load_model("video.h5")
predict_video("/content/drive/MyDrive/Colab Notebooks/ય3.mp4", model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')